# 240 · Same readout pulse, different phase

**Level:** medium · **Topics:** Gen2, SHFQA, readout, phase, device-setup, calibration · **Type:** fix the bugs

A user wants to send the same readout pulse twice in one shot, the second time with a phase of 90°.
Passing `phase=` to the second pulse looks like the obvious way, but the experiment does not compile.

Make it work: same pulse shape, same frequency, same output, second pulse shifted by 90°.

Everything runs in emulation, no instrument needed.

In [ ]:
import numpy as np

from laboneq.simple import *
from laboneq.simulator.output_simulator import OutputSimulator

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Experiment

Two readout pulses, each with its own acquisition. The second one has a phase of 90°.

📖 [Amplitude and phase of pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/06_pulses_amplitude_and_phase.html) ·
[Measurement rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/03_measurement_rules.html)

In [ ]:
readout_pulse = pulse_library.const(length=1e-6, amplitude=0.5)

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def two_readouts():
    with dsl.acquire_loop_rt(count=16):
        with dsl.section(name="first") as first:
            dsl.play("measure", readout_pulse)
            dsl.acquire("acquire", handle="first", kernel=readout_pulse)
            dsl.delay("measure", time=1e-6)
        with dsl.section(name="second", play_after=first):
            dsl.play("measure", readout_pulse, phase=np.pi / 2)
            dsl.acquire("acquire", handle="second", kernel=readout_pulse)
            dsl.delay("measure", time=1e-6)


experiment = two_readouts()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Compile and run

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

## Check

The cell below reads the simulated readout output in the middle of each pulse (0.5 µs and 2.5 µs) and prints the phase between the two.

📖 [Output simulator](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/10_advanced_topics/tutorials/00_output_simulator.html)

In [ ]:
# check
simulation = OutputSimulator(compiled)
channel = setup.logical_signal_groups["q0"].logical_signals["measure"].physical_channel
snippet = simulation.get_snippet(channel, start=0, output_length=4e-6)

first = snippet.wave[np.argmin(abs(snippet.time - 0.5e-6))]
second = snippet.wave[np.argmin(abs(snippet.time - 2.5e-6))]
print(round(np.degrees(np.angle(second / first))), "degrees")

**Expected result:** `90 degrees`

## Questions

1. What does the error say, and why does a phase count as a different pulse here?
2. The same `phase=` on the `drive` line works. Why?
3. Name two ways to get the two phases on the readout line.

<details>
<summary>Hint 1</summary>

A readout output stores one waveform per logical signal. More than one logical signal can be connected to the same port.
</details>

<details>
<summary>Hint 2</summary>

Three places need the new signal: the device setup, the calibration, and the experiment with its signal map.
</details>

The solution is in [solution.ipynb](solution.ipynb).